# Monarka ML · Patrones y outfits

Demo educativa con **datos sintéticos**. Reemplaza los CSV por catálogo y valoraciones humanas para evaluar la utilidad real. El modelo aprende compatibilidad de pares y DFS construye conjuntos. El clustering explora grupos del catálogo.

## Preparar el entorno
En Colab se clona el proyecto y se instalan dependencias. En local, abre el notebook desde la raíz del repositorio con el entorno activado.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repo = Path("/content/Monarca-ML")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "https://github.com/Oscar-h23/Monarca-ML.git", str(repo)])
    os.chdir(repo)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
elif not Path("src").exists() and Path("../src").exists():
    os.chdir("..")
assert Path("src").exists(), "Ejecuta desde la carpeta del repositorio."

## 1. Pandas y etiquetas
Una fila del catálogo representa un producto y color, agrupando las tallas. Las valoraciones se promedian por par. El umbral de clasificación será 4/5.

In [ ]:
from src.preparar_datos import cargar_datos, tabla_pares, dividir_productos
catalogo, pares = cargar_datos("data/demo/catalogo.csv", "data/demo/valoraciones.csv")
display(catalogo.head())
tabla = tabla_pares(catalogo, pares)
display(tabla.head())
print("Productos", catalogo.id_producto.nunique(), "Pares", len(tabla))

## 2. NumPy, patrones y clustering
Revisa media, mediana, dispersión y atípicos de precios. KMeans usa variables codificadas y precio escalado. Los números de cluster son identificadores; describe cada grupo a partir de sus filas. Los grupos no determinan si dos prendas combinan.

In [ ]:
from src.analizar_patrones import analizar
estadisticas, clusters, patrones = analizar(catalogo, tabla)
display(estadisticas)
display(clusters.groupby("cluster").agg(prendas=("id_producto", "size"), precio_medio=("precio", "mean")))
display(patrones.sort_values("ejemplos", ascending=False).head(10))

## 3. Separar por producto
Reservamos 60% entrenamiento, 20% validación y 20% prueba. Se excluyen pares que cruzan particiones. Todos los colores de cada producto quedan juntos.

In [ ]:
tablas, grupos = dividir_productos(catalogo, pares, semilla=23)
display({nombre: {"productos": len(grupos[nombre]), "pares": len(t), "positivos": int((t.valoracion >= 4).sum())} for nombre, t in tablas.items()})

## 4. Clasificación y regresión
El árbol y la logística predicen recomendable/poco recomendable. Elegimos por F1 de validación y reportamos prueba. La regresión lineal predice una puntuación y se compara con la media. Analiza los baselines antes de concluir que el modelo aporta.

In [ ]:
from src.entrenar import entrenar
clasificador, regresor, metricas, reglas = entrenar(tablas, semilla=23)
display(metricas)
print(reglas)

## 5. Grafo y DFS
Después de evaluar reajustamos los modelos seleccionados con todas las etiquetas. Cada nodo es producto y color. DFS considera categorías distintas y comprueba las tres aristas de cada conjunto. El ranking usa el promedio de puntuaciones estimadas, recortadas a 1–5.

In [ ]:
from src.generar_outfits import generar
clasificador.fit(tabla, tabla.valoracion >= 4)
regresor.fit(tabla, tabla.valoracion)
recomendaciones = generar(catalogo, clasificador, regresor)
ejemplo = next((r for r in recomendaciones if r["outfits"]), None)
display(ejemplo)
if ejemplo:
    piezas = ejemplo["outfits"][0]["prendas"]
    display(catalogo.loc[[any(row.id_producto == p["idProducto"] and row.color == p["color"] for p in piezas) for row in catalogo.itertuples()]])

## 6. Exportar para Monarka
El comando reproduce el análisis y guarda métricas, gráficos y outfits. Spring Boot debe verificar catálogo y stock actuales antes de responder. Consulta docs/integracion.md.

In [ ]:
from src.ejecutar import ejecutar
ejecutar("data/demo/catalogo.csv", "data/demo/valoraciones.csv", demo=True)
from IPython.display import Image
display(Image(filename="resultados/analisis.png"))

## Discusión de resultados
- ¿El modelo supera a la clase mayoritaria y a la regla del mismo estilo?
- ¿Qué sesgos introducen las etiquetas del equipo?
- ¿Qué clusters mezclan estilos o categorías?
- ¿Los evaluadores prefieren los conjuntos completos frente al baseline?
- ¿Qué cambia al incorporar valoraciones reales?

Las métricas de esta demo describen la regla sintética. Para analizar patrones de compra se requiere incorporar pedidos confirmados y un protocolo de evaluación adicional.